In [1]:
import json
import os
from pathlib import Path
from pprint import pprint
import numpy as np

from datasets import load_from_disk
from transformers import AutoModelForTokenClassification, AutoTokenizer,\
DataCollatorForTokenClassification, TrainingArguments, Trainer, EvalPrediction
import evaluate

import wandb
from colorama import Fore, Style

In [2]:
import torch
torch.cuda.empty_cache()

In [3]:
from transformers.models.deberta_v2.tokenization_deberta_v2 import DebertaV2Tokenizer
from transformers.models.deberta_v2.modeling_deberta_v2 import DebertaV2ForTokenClassification
from datasets import DatasetDict

In [4]:
model_path = "FacebookAI/roberta-base"
# model_path = "microsoft/deberta-v3-small"
# model_path = "distilbert/distilbert-base-uncased" #temp

In [5]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)

LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODEL_DIR = Path(os.getcwd()).parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
RUN_NAME = "pii_redaction_" + model_path.replace("/", "_") + "_v1"

In [6]:
dataset: DatasetDict = load_from_disk(DATASET_PATH) # type:ignore
print(f"dataset column names: {dataset["train"].column_names}\n\n")

dataset column names: ['source_text', 'privacy_mask']




In [7]:
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
label2id = label_info["label2id"]
id2label = label_info["id2label"]

In [8]:
tokenizer: DebertaV2Tokenizer = AutoTokenizer.from_pretrained(model_path)
model: DebertaV2ForTokenClassification = AutoModelForTokenClassification.from_pretrained(
    model_path, num_labels=len(id2label), id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
print(type(model))
print(type(tokenizer))

<class 'transformers.models.roberta.modeling_roberta.RobertaForTokenClassification'>
<class 'transformers.models.roberta.tokenization_roberta.RobertaTokenizer'>


In [10]:
assert tokenizer.is_fast

In [11]:
example = dataset["train"][0]
print("dataset example:")
for item in example.items():
    print(item)

dataset example:
('source_text', 'Subject: Group Messaging for Admissions Process\n\nGood morning, everyone,\n\nI hope this message finds you well. As we continue our admissions processes, I would like to update you on the latest developments and key information. Please find below the timeline for our upcoming meetings:\n\n- wynqvrh053 - Meeting at 10:20am\n- luka.burg - Meeting at 21\n- qahil.wittauer - Meeting at quarter past 13\n- gholamhossein.ruschke - Meeting at 9:47 PM\n- pdmjrsyoz1460 ')
('privacy_mask', [{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}, {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}, {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}, {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}, {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}, {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}, {'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'la

In [12]:
ex_tokenized_with_offsets = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
for item in ex_tokenized_with_offsets.items():
    print(item)

('input_ids', [0, 47159, 35, 826, 15212, 6257, 13, 1614, 23363, 19149, 50118, 50118, 12350, 662, 6, 961, 6, 50118, 50118, 100, 1034, 42, 1579, 5684, 47, 157, 4, 287, 52, 535, 84, 18054, 5588, 6, 38, 74, 101, 7, 2935, 47, 15, 5, 665, 5126, 8, 762, 335, 4, 3401, 465, 874, 5, 10589, 13, 84, 2568, 2891, 35, 50118, 50118, 12, 885, 3892, 22638, 20378, 2546, 246, 111, 12776, 23, 158, 35, 844, 424, 50118, 12, 784, 10620, 4, 3321, 111, 12776, 23, 733, 50118, 12, 2231, 895, 718, 4, 605, 2582, 9994, 111, 12776, 23, 297, 375, 508, 50118, 12, 821, 9649, 424, 298, 366, 36353, 4, 14888, 611, 1071, 111, 12776, 23, 361, 35, 3706, 2784, 50118, 12, 181, 43604, 267, 338, 8628, 3979, 1570, 2466, 1437, 2])
('attention_mask', [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [13]:
tokens = ex_tokenized_with_offsets.tokens()
print(tokens)

['<s>', 'Subject', ':', 'ĠGroup', 'ĠMess', 'aging', 'Ġfor', 'ĠAd', 'missions', 'ĠProcess', 'Ċ', 'Ċ', 'Good', 'Ġmorning', ',', 'Ġeveryone', ',', 'Ċ', 'Ċ', 'I', 'Ġhope', 'Ġthis', 'Ġmessage', 'Ġfinds', 'Ġyou', 'Ġwell', '.', 'ĠAs', 'Ġwe', 'Ġcontinue', 'Ġour', 'Ġadmissions', 'Ġprocesses', ',', 'ĠI', 'Ġwould', 'Ġlike', 'Ġto', 'Ġupdate', 'Ġyou', 'Ġon', 'Ġthe', 'Ġlatest', 'Ġdevelopments', 'Ġand', 'Ġkey', 'Ġinformation', '.', 'ĠPlease', 'Ġfind', 'Ġbelow', 'Ġthe', 'Ġtimeline', 'Ġfor', 'Ġour', 'Ġupcoming', 'Ġmeetings', ':', 'Ċ', 'Ċ', '-', 'Ġw', 'yn', 'qv', 'rh', '05', '3', 'Ġ-', 'ĠMeeting', 'Ġat', 'Ġ10', ':', '20', 'am', 'Ċ', '-', 'Ġl', 'uka', '.', 'burg', 'Ġ-', 'ĠMeeting', 'Ġat', 'Ġ21', 'Ċ', '-', 'Ġq', 'ah', 'il', '.', 'w', 'itt', 'auer', 'Ġ-', 'ĠMeeting', 'Ġat', 'Ġquarter', 'Ġpast', 'Ġ13', 'Ċ', '-', 'Ġg', 'hol', 'am', 'h', 'os', 'sein', '.', 'rus', 'ch', 'ke', 'Ġ-', 'ĠMeeting', 'Ġat', 'Ġ9', ':', '47', 'ĠPM', 'Ċ', '-', 'Ġp', 'dm', 'j', 'r', 'sy', 'oz', '14', '60', 'Ġ', '</s>']


In [14]:
word_ids = ex_tokenized_with_offsets.word_ids()
print(word_ids)

[None, 0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 58, 58, 58, 59, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 78, 78, 79, 80, 80, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 89, 89, 89, 89, 89, 90, 91, 91, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 101, 101, 101, 101, 101, 102, 102, 103, None]


In [15]:
pprint(example["privacy_mask"], sort_dicts=False)

[{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'},
 {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'},
 {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'},
 {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'},
 {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'},
 {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'},
 {'value': 'gholamhossein.ruschke',
  'start': 395,
  'end': 416,
  'label': 'USERNAME'},
 {'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'},
 {'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}]


In [16]:
def get_ner_labels(batch: dict[str, list]) -> dict[str, list[list[str]]]:
    token_offsets = tokenizer(
        batch["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True
    )["offset_mapping"]

    batch_ner_labels: list[list[str]] = []
    # loop through the batch to get the privacy masks for every sequence
    for i, row_masks in enumerate(batch["privacy_mask"]):
        row_ner_labels = []
        # loop through the token_offsets of the current sequence
        for offset in token_offsets[i]:
            # if add_special_tokens is true skip over special tokens (offset with (0,0))
            if offset == (0, 0): 
                row_ner_labels.append("O")
                continue
            # label is "O" unless privacy label is found
            label = "O" 
            # loop through masks to find label
            for privacy_mask in row_masks:
                # if statement is switched to check if any character of the token falls within the mask
                if offset[1] > privacy_mask["start"] and offset[0] < privacy_mask["end"]:
                    label = privacy_mask["label"]
                    # switch if statement to also include less than
                    if offset[0] <= privacy_mask["start"]:
                        label = "B-" + label
                    else:
                        label = "I-" + label
                    
                    break # break out of for loop when/if label is found
            row_ner_labels.append(label)
            
        batch_ner_labels.append(row_ner_labels)
    
    return {"ner_tags": batch_ner_labels}

In [17]:
dataset = dataset.map(get_ner_labels, batched=True, batch_size=20_000)
pprint(dataset["train"].features)

{'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string')}


In [18]:
def tokenize_and_align_labels_single(example: dict[str, list]):
    tokenized_input = tokenizer(
        example["source_text"],
        truncation=True,
        add_special_tokens=True
    )
    labels = []
    word_ids = tokenized_input.word_ids()
    previous_word_idx = None
    for i, tag in enumerate(example["ner_tags"]):
        word_idx = word_ids[i]
        
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(label2id[tag])
        else:
            labels.append(-100)
        previous_word_idx = word_idx
            
    tokenized_input["labels"] = labels
    return tokenized_input
            

In [19]:
dataset = dataset.map(tokenize_and_align_labels_single, batched=False)

In [20]:
pprint(dataset["train"].features)

{'attention_mask': List(Value('int8')),
 'input_ids': List(Value('int32')),
 'labels': List(Value('int64')),
 'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string')}


In [21]:
example = dataset["train"][0]
print(f"{len(example["ner_tags"])=}")
print(f"{len(example["input_ids"])=}")

len(example["ner_tags"])=130
len(example["input_ids"])=130


In [22]:
ex_wi = tokenizer(
    example["source_text"],
    truncation=True,
    add_special_tokens=True
).word_ids()
masks = example["privacy_mask"]

token_offsets = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)["offset_mapping"]

id2label = {i: label for label, i in label2id.items()}
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])

In [23]:
for mask in masks:
    print(Fore.BLUE + f"{mask}" + Style.RESET_ALL)    
for tag, wi, label in zip(example["labels"], ex_wi, token_offsets):
    if tag != -100 and tag != 0:
        token = tokens[wi] #type:ignore
        value = f"label={id2label[tag]}, word_id={wi}, {token=}, {label=}"
        print(value)
        
        masks_lst = []
        for mask in masks:
            if label[1] > mask["start"] and label[0] < mask["end"]:
                masks_lst.append(f"in {mask}")
                
        if masks_lst:
            print(Fore.GREEN + f"{masks_lst}")
            print(Style.RESET_ALL + "", end="")
        else:
            print(Fore.RED + f" {label} not in masks" )
            print(Style.RESET_ALL + "", end="")

{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}
{'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}
{'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}
{'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}
{'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}
{'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}
{'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'label': 'USERNAME'}
{'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'}
{'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}
label=B-USERNAME, word_id=58, token='Ċ', label=(287, 288)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=I-USERNAME, word_id=59, token='Ċ', label=(294, 296)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=B-TIME, word_id=63, token='qv', label=(311, 313)
["in {'value': '10:20am', 'start': 311, 'e

In [24]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, word_id, label in zip(example_encodings.tokens(), example["ner_tags"], example_encodings.word_ids(), example_encodings["offset_mapping"]):
    tag = tag
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(word_id), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += word_id + " " * (max_length - len(word_id) + 1)
    line4 += label + " " * (max_length - len(label) + 1)
print(line1)
print(line2)
print(line3)
print(line4)

<s>    Subject :      ĠGroup  ĠMess    aging    Ġfor     ĠAd      missions ĠProcess Ċ        Ċ        Good     Ġmorning ,        Ġeveryone ,        Ċ        Ċ        I        Ġhope    Ġthis    Ġmessage Ġfinds   Ġyou       Ġwell      .          ĠAs        Ġwe        Ġcontinue  Ġour       Ġadmissions Ġprocesses ,          ĠI         Ġwould     Ġlike      Ġto        Ġupdate    Ġyou       Ġon        Ġthe       Ġlatest    Ġdevelopments Ġand       Ġkey       Ġinformation .          ĠPlease    Ġfind      Ġbelow     Ġthe       Ġtimeline  Ġfor       Ġour       Ġupcoming  Ġmeetings  :          Ċ          Ċ          -          Ġw         yn         qv         rh         05         3          Ġ-         ĠMeeting   Ġat        Ġ10        :          20         am         Ċ          -          Ġl         uka        .          burg       Ġ-         ĠMeeting   Ġat        Ġ21        Ċ          -          Ġq         ah         il         .          w          itt        auer       Ġ-         ĠMeeting   Ġa

In [25]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, label, word_id in zip(example_encodings.tokens(), example["ner_tags"], example["labels"], example_encodings.word_ids()):
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += label + " " * (max_length - len(label) + 1)
    line4 += word_id + " " * (max_length - len(word_id) + 1)
print(line1)
print(line2)
print(line3)
print(line4)

<s>  Subject : ĠGroup ĠMess aging Ġfor ĠAd missions ĠProcess Ċ Ċ Good Ġmorning , Ġeveryone , Ċ Ċ I Ġhope Ġthis Ġmessage Ġfinds Ġyou Ġwell . ĠAs Ġwe Ġcontinue Ġour Ġadmissions Ġprocesses , ĠI Ġwould Ġlike Ġto Ġupdate Ġyou Ġon Ġthe Ġlatest Ġdevelopments Ġand Ġkey Ġinformation . ĠPlease Ġfind Ġbelow Ġthe Ġtimeline Ġfor Ġour Ġupcoming Ġmeetings : Ċ Ċ - Ġw         yn         qv         rh         05         3          Ġ- ĠMeeting Ġat Ġ10    :      20     am     Ċ - Ġl         uka        .          burg       Ġ- ĠMeeting Ġat Ġ21    Ċ - Ġq         ah         il         .          w          itt        auer       Ġ- ĠMeeting Ġat Ġquarter Ġpast  Ġ13    Ċ - Ġg         hol        am         h          os         sein       .          rus        ch         ke         Ġ- ĠMeeting Ġat Ġ9     :      47     ĠPM    Ċ - Ġp         dm         j          r          sy         oz         14         60         Ġ </s> 
O    O       O O      O     O     O    O   O        O        O O O    O        O O        

In [26]:
print([label for label in example["labels"]])

[-100, 0, 0, 0, 0, -100, 0, 0, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -100, -100, -100, 2, -100, 0, 0, 0, 3, 4, 4, 4, 0, 0, 1, -100, 2, 2, 0, 0, 0, 3, 0, 0, 1, -100, -100, 2, 2, -100, -100, 0, 0, 0, 3, 4, 4, 0, 0, 1, -100, -100, -100, -100, -100, 2, 2, -100, -100, 0, 0, 0, 3, 4, 4, 4, 0, 0, 1, -100, -100, -100, -100, -100, 2, -100, 0, -100]


In [27]:
def validate_label_count(dataset):
    valid = []
    for i in range(len(dataset)):
        labels = dataset[i]["labels"]
        masks = dataset[i]["privacy_mask"]
        b_entities = [label for label in labels if label != -100 or label != 0]
        valid.append(len(b_entities) == len(masks))
    return valid

pprint(validate_label_count(dataset["train"].select(range(5))))

[False, False, False, False, False]


In [28]:
def check_label_distribution(dataset, num_samples=5):
    for i in range(num_samples):
        labels = dataset[i]["labels"]
        valid = [label for label in labels if label != -100]
        print(f"Sample {i}: {len(valid)} valid labels out of {len(labels)} tokens")
        print(f"  Labels: {valid[:10]}")

check_label_distribution(dataset["train"])

Sample 0: 104 valid labels out of 130 tokens
  Labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sample 1: 95 valid labels out of 101 tokens
  Labels: [0, 0, 0, 3, 4, 4, 4, 0, 0, 1]
Sample 2: 100 valid labels out of 121 tokens
  Labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sample 3: 139 valid labels out of 182 tokens
  Labels: [0, 0, 15, 16, 16, 0, 0, 0, 0, 0]
Sample 4: 89 valid labels out of 103 tokens
  Labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [29]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

In [30]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zelluzy/.netrc.
wandb: Currently logged in as: bengid (bengid-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [31]:
from collections import Counter
def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    predictions, label_ids = p.predictions, p.label_ids
    predictions = np.argmax(predictions, axis=-1)
    
    true_preds = [
        [id2label[pred] for pred, tgt_id in zip(preds_row, labels_row) if tgt_id != -100]
        for preds_row, labels_row in zip(predictions, label_ids)
    ]
    
    true_labels =[
        [id2label[tgt_id] for tgt_id in labels_row if tgt_id != -100]
        for labels_row in label_ids
    ]
    
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    
    flat = {}
    if results is not None:
        flat.update({
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        })
        
        for entity, scores in results.items():
            if isinstance(scores, dict): # filter scores for individual labels
                flat[f"{entity}_f1"] = scores["f1"]
                flat[f"{entity}_support"] = scores["number"]
    
    return flat

In [32]:
# small_train = dataset["train"].select(range(32))

# temp_args = TrainingArguments(
#     output_dir=str(MODEL_DIR),
#     learning_rate=1e-5, # low learning_rate
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=1,
#     weight_decay=0.01,
#     eval_strategy="epoch",
#     warmup_ratio=0.1,
#     bf16=False,
#     fp16=False,
#     max_grad_norm=1.0,
# )

# temp_trainer = Trainer(
#     model=model,
#     args=temp_args,
#     train_dataset=small_train,
#     eval_dataset=dataset["validation"],
#     processing_class=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
# )
# print(temp_trainer.args.device)
# trainer_output = temp_trainer.train()

In [33]:
run = wandb.init(project="pii-redaction", name="roberta_bf16")

In [ ]:
num_train_samples = len(dataset["train"])   # 29908
batch_size = 16
num_epochs = 1
total_steps = (num_train_samples // batch_size) * num_epochs  # 1869
warmup_steps = int(0.1 * total_steps)  # 186

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    report_to="wandb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    warmup_steps=warmup_steps,
    bf16=True,
    tf32=False,
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [35]:
print(trainer.args.device)

cuda:0


In [36]:
trainer_output = trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Bod F1,Bod Support,Building F1,Building Support,City F1,City Support,Country F1,Country Support,Date F1,Date Support,Driverlicense F1,Driverlicense Support,Email F1,Email Support,Geocoord F1,Geocoord Support,Givenname1 F1,Givenname1 Support,Givenname2 F1,Givenname2 Support,Idcard F1,Idcard Support,Ip F1,Ip Support,Lastname1 F1,Lastname1 Support,Lastname2 F1,Lastname2 Support,Lastname3 F1,Lastname3 Support,Pass F1,Pass Support,Passport F1,Passport Support,Postcode F1,Postcode Support,Secaddress F1,Secaddress Support,Sex F1,Sex Support,Socialnumber F1,Socialnumber Support,State F1,State Support,Street F1,Street Support,Tel F1,Tel Support,Time F1,Time Support,Title F1,Title Support,Username F1,Username Support
1,0.073579,0.041979,0.924399,0.935283,0.929809,0.991053,0.960345,1148,0.976861,987,0.965585,1005,0.956913,765,0.907918,838,0.941028,1201,0.981337,1272,0.975845,104,0.791740,954,0.622517,268,0.918529,1352,0.991404,1046,0.755126,1211,0.463245,318,0.397059,106,0.952440,804,0.924757,1213,0.984456,959,0.981735,442,0.964411,979,0.946926,1316,0.972842,1020,0.964249,971,0.953253,1037,0.980892,1876,0.961880,957,0.948894,1331
2,0.043824,0.039666,0.935162,0.948705,0.941885,0.991768,0.963948,1148,0.981266,987,0.968950,1005,0.964676,765,0.905569,838,0.928033,1201,0.985191,1272,0.971429,104,0.857446,954,0.720648,268,0.929825,1352,0.983309,1046,0.823211,1211,0.662921,318,0.402597,106,0.949297,804,0.909303,1213,0.977892,959,0.972851,442,0.972264,979,0.958957,1316,0.980966,1020,0.973028,971,0.956728,1037,0.979505,1876,0.973172,957,0.962158,1331
3,0.035899,0.033699,0.940809,0.956279,0.948481,0.992471,0.969486,1148,0.982846,987,0.972387,1005,0.968072,765,0.932709,838,0.952782,1201,0.976998,1272,0.976077,104,0.878955,954,0.750424,268,0.927411,1352,0.988539,1046,0.822101,1211,0.692547,318,0.649215,106,0.943602,804,0.933065,1213,0.974936,959,0.977324,442,0.976227,979,0.956893,1316,0.984871,1020,0.975064,971,0.970448,1037,0.980774,1876,0.972091,957,0.966554,1331
4,0.030518,0.032717,0.945928,0.957771,0.951813,0.992782,0.969197,1148,0.984336,987,0.977690,1005,0.973146,765,0.923451,838,0.958961,1201,0.980867,1272,0.980769,104,0.887089,954,0.793651,268,0.936264,1352,0.990458,1046,0.833948,1211,0.703947,318,0.672986,106,0.947048,804,0.935419,1213,0.980453,959,0.981818,442,0.976203,979,0.961598,1316,0.988258,1020,0.973083,971,0.959466,1037,0.983790,1876,0.974763,957,0.963684,1331
5,0.023380,0.030996,0.948741,0.958124,0.953409,0.993432,0.970013,1148,0.985361,987,0.977690,1005,0.972454,765,0.937685,838,0.954052,1201,0.978524,1272,0.976077,104,0.892086,954,0.798521,268,0.940916,1352,0.990440,1046,0.841847,1211,0.713355,318,0.699552,106,0.939046,804,0.935002,1213,0.984997,959,0.981818,442,0.973604,979,0.965677,1316,0.984871,1020,0.978528,971,0.969319,1037,0.981944,1876,0.973228,957,0.965414,1331
6,0.020955,0.031868,0.949014,0.959144,0.954052,0.993372,0.969987,1148,0.984864,987,0.976721,1005,0.970551,765,0.931198,838,0.953615,1201,0.979353,1272,0.980769,104,0.892894,954,0.801457,268,0.943089,1352,0.990440,1046,0.843137,1211,0.733553,318,0.703704,106,0.940109,804,0.936514,1213,0.985537,959,0.982935,442,0.973658,979,0.966654,1316,0.985380,1020,0.977505,971,0.967557,1037,0.982233,1876,0.974763,957,0.969103,1331


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [37]:
for i in trainer_output:
    pprint(i)

11220
0.06227331414367213
{'epoch': 6.0,
 'total_flos': 2.596632051418488e+16,
 'train_loss': 0.06227331414367213,
 'train_runtime': 984.8062,
 'train_samples_per_second': 182.217,
 'train_steps_per_second': 11.393}


In [38]:
out = trainer.predict(dataset["test"]) #type:ignore
preds = np.argmax(out.predictions, axis=-1)
print(preds)
print(Fore.CYAN + "")
pprint(out.metrics)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 7 8 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

{'test_BOD_f1': 0.9754028837998303,
 'test_BOD_support': 1168,
 'test_BUILDING_f1': 0.9869315211709356,
 'test_BUILDING_support': 956,
 'test_CITY_f1': 0.9763313609467457,
 'test_CITY_support': 1012,
 'test_COUNTRY_f1': 0.9740740740740741,
 'test_COUNTRY_support': 800,
 'test_DATE_f1': 0.9382573571840739,
 'test_DATE_support': 864,
 'test_DRIVERLICENSE_f1': 0.9546578730420445,
 'test_DRIVERLICENSE_support': 1219,
 'test_EMAIL_f1': 0.9910647803425168,
 'test_EMAIL_support': 1340,
 'test_GEOCOORD_f1': 0.9865470852017937,
 'test_GEOCOORD_support': 112,
 'test_GIVENNAME1_f1': 0.8940092165898618,
 'test_GIVENNAME1_support': 1064,
 'test_GIVENNAME2_f1': 0.7871939736346516,
 'test_GIVENNAME2_support': 271,
 'test_IDCARD_f1': 0.9418913786555261,
 'test_IDCARD_support': 1300,
 'test_IP_f1': 0.9937388193202147,
 'test_IP_support': 1120,
 'test_LASTNAME1_f1': 0.8528209321340964

In [39]:
import torch
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")
        

In [40]:
import html as html_lib

def log_pii_test_results(
    trainer,
    test_dataset,
    tokenizer,
    table_name: str = "pii_test_results",
    run: Optional[wandb.sdk.wandb_run.Run] = None,
) -> wandb.Table:
    # Bug 1 fix: resolve to a concrete run object, not the wandb module.
    # wandb.run is the currently active run (set by wandb.init or the Trainer callback).
    active_run = run or wandb.run
    if active_run is None:
        raise RuntimeError(
            "No active W&B run found. Either pass `run=` explicitly "
            "or call wandb.init() before invoking this function."
        )

    # Bug 2 fix: normalize to int keys — live models use int, JSON-reloaded models use str.
    id2label = {int(k): v for k, v in trainer.model.config.id2label.items()}

    pred_output = trainer.predict(test_dataset)
    pred_ids = np.argmax(pred_output.predictions, axis=-1)
    true_ids = pred_output.label_ids

    table = wandb.Table(columns=["id", "annotated_sequence", "f1", "precision", "recall", "tp", "fp", "fn"])

    for i in range(len(test_dataset)):
        tokens, true_labels, pred_labels = [], [], []
        for token_id, true_id, pred_id in zip(test_dataset[i]["input_ids"], true_ids[i], pred_ids[i]):
            if true_id == -100:
                continue
            tokens.append(tokenizer.convert_ids_to_tokens(int(token_id)))
            true_labels.append(id2label[int(true_id)])
            pred_labels.append(id2label[int(pred_id)])

        tp = fp = fn = 0
        for true, pred in zip(true_labels, pred_labels):
            if true != "O" and pred == true:   tp += 1
            elif true != "O" and pred != true: fn += 1
            elif true == "O" and pred != "O":  fp += 1

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        parts = []
        for token, true, pred in zip(tokens, true_labels, pred_labels):
            t = html_lib.escape(token)
            if true != "O" and pred == true:
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#C0DD97;color:#27500A">{html_lib.escape(true)}</sup>'
                parts.append(f'<span style="background:#C0DD97;color:#27500A;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            elif true != "O" and pred != true:
                label = html_lib.escape(f"{true}→{pred}")
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#F7C1C1;color:#791F1F">{label}</sup>'
                parts.append(f'<span style="background:#F7C1C1;color:#791F1F;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            elif true == "O" and pred != "O":
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#FAC775;color:#633806">{html_lib.escape(pred)}</sup>'
                parts.append(f'<span style="background:#FAC775;color:#633806;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            else:
                parts.append(f'<span style="margin:1px 2px;display:inline-block">{t}</span>')

        seq_html = '<div style="font-family:monospace;font-size:12px;line-height:2">' + "".join(parts) + "</div>"
        table.add_data(i, wandb.Html(seq_html), round(f1, 4), round(precision, 4), round(recall, 4), tp, fp, fn)

    active_run.log({table_name: table})
    return table

In [41]:
log_pii_test_results(trainer, dataset["test"], tokenizer)

In [42]:
small_test = dataset["test"].select(range(32))

In [43]:
pred_output = trainer.predict(small_test)
arr = pred_output.predictions

u = np.unique(pred_output.predictions)
print(u)

[-100.        -4.25      -4.21875 ...   13.1875    13.25      13.3125 ]


In [44]:
pred_output = trainer.predict(
    test_dataset=dataset["test"]
)
f1s = []
for metric, value in pred_output.metrics.items():
    if metric.endswith("f1"):
        f1s.append((metric, value))
        
sorted_f1s = sorted(f1s, key=lambda x: x[1], reverse=True)

print("worst f1s:")
for metric, value in reversed(sorted_f1s[-7:]):
    print(f"{metric:<20}= {value:.3f}")

worst f1s:
test_LASTNAME3_f1   = 0.724
test_LASTNAME2_f1   = 0.766
test_GIVENNAME2_f1  = 0.787
test_LASTNAME1_f1   = 0.853
test_GIVENNAME1_f1  = 0.894
test_PASS_f1        = 0.935
test_DATE_f1        = 0.938


In [45]:
wandb.finish()

eval/BOD_f1,▁▄█▇██
eval/BOD_support,▁▁▁▁▁▁
eval/BUILDING_f1,▁▅▆▇██
eval/BUILDING_support,▁▁▁▁▁▁
eval/CITY_f1,▁▃▅██▇
eval/CITY_support,▁▁▁▁▁▁
eval/COUNTRY_f1,▁▄▆██▇
eval/COUNTRY_support,▁▁▁▁▁▁
eval/DATE_f1,▂▁▇▅█▇
eval/DATE_support,▁▁▁▁▁▁
+119,...
